In [1]:
from codecs import xmlcharrefreplace_errors

from data_generation import Environment
from eswm import ESWM,get_batch,OpenWorld,accuracy,RandomWall
import torch

In [2]:
open_arena = Environment()
random_wall= Environment(side_length=4,add_wall=True,hidden=5,possible_states=37)
#ds =env.generate_memory_bank(128,True)

In [3]:
open_embed = OpenWorld()
random_embed = RandomWall()

In [4]:
device = torch.device('mps')
eswm =ESWM(embedder=open_embed)
#eswm = ESWM(embedder=random_embed,state_dim=37,input_dim=1024)
x,y,mx,mp =get_batch(open_arena,128)
eswm.to(device)
x=x.to(device)
y=[yi.to(device) for yi in y]
mx=mx.to(device)
mp=mp.to(device)
out=eswm(x,mp,mx)
#p = [out[i] for i in range(3)]
#t = [y[i] for i in range(3)]
#m = torch.stack([mx]*6).permute([2,1,0])
m =[mx[:,0:6],mx[:,6:7],mx[:,7:]]
#p = out.masked_select(mx).view(-1, out.shape[1])
#t= y.masked_select(mx).view(-1, y.shape[1])
p = [torch.masked_select(out[i], m[i]) for i in range(3)]
t = [torch.masked_select(y[i], m[i]).view(-1, m[i].shape[1]) for i in range(3)]

In [26]:
model=ESWM(open_embed,num_layers=6)
state_dict = torch.load('test.pth',map_location='cpu',weights_only=True)
model.load_state_dict(state_dict)
x,y,query_mask,padding_mask = get_batch(open_arena,batch_size=128,test=True)
model.to(torch.device('cpu'))
model.eval()
output= model(x,padding_mask,query_mask)

m = [query_mask[:, 0:6], query_mask[:, 6:7].repeat(1,6), query_mask[:, 7:]]
p = [torch.masked_select(output[i], m[i].ne(0)).view(-1,6) for i in range(3)]
t = [torch.masked_select(y[i], m[i][:,:y[i].shape[1]]).view(-1,y[i].shape[1]) for i in range(3)]

acc = accuracy(p,t)


In [27]:
p[1].argmax(dim=1)


tensor([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])

In [25]:
torch.nn.functional.one_hot(t[1][:,0].long(), 6).shape


torch.Size([40, 6])

In [29]:
x[:,-1,6]


tensor([3, 0, 1, 5, 0, 5, 0, 4, 5, 1, 4, 5, 3, 0, 4, 0, 1, 5, 5, 2, 0, 4, 3, 2,
        2, 4, 4, 3, 4, 0, 2, 2, 0, 0, 2, 2, 1, 2, 4, 0, 2, 5, 1, 5, 4, 5, 3, 5,
        4, 5, 0, 0, 0, 2, 5, 0, 1, 2, 1, 0, 2, 0, 0, 2, 3, 1, 5, 1, 4, 3, 1, 1,
        1, 3, 4, 0, 5, 2, 3, 4, 2, 5, 3, 2, 5, 5, 4, 0, 1, 5, 5, 1, 2, 2, 4, 2,
        5, 3, 4, 4, 5, 4, 1, 4, 2, 4, 2, 5, 4, 5, 5, 0, 5, 4, 1, 5, 3, 4, 2, 2,
        1, 1, 0, 4, 2, 4, 3, 0])